# Amazon Supply Chain Intelligence
## Exploratory Data Analysis & Preprocessing

**Author:** ARUN AK  
**Course:** UCS321 - AI for Engineers  
**Project:** Delivery Delay Risk Prediction

---

### Table of Contents
1. [Data Loading & Overview](#1-data-loading)
2. [Data Quality Assessment](#2-data-quality)
3. [Univariate Analysis](#3-univariate)
4. [Bivariate Analysis](#4-bivariate)
5. [Feature Engineering](#5-feature-engineering)
6. [Target Variable Analysis](#6-target-analysis)
7. [Key Insights & Findings](#7-insights)

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('Libraries loaded successfully!')

---
## 1. Data Loading & Overview <a id='1-data-loading'></a>

In [ ]:
# Load the dataset
df = pd.read_csv('../data/raw/amazon_delivery.csv')

print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# First look at the data
df.head(10)

In [ ]:
# Data types and info
print("=" * 50)
print("DATA TYPES")
print("=" * 50)
df.info()

In [ ]:
# Statistical summary
print("=" * 50)
print("NUMERICAL FEATURES SUMMARY")
print("=" * 50)
df.describe().round(2)

### Key Observations:
- **43,739 delivery records** from Amazon's last-mile logistics
- **16 features** including agent info, location, weather, traffic
- **Delivery Time** ranges from 10 to 270 minutes (target variable basis)
- Agent ages range from 20-40 years, ratings from 1.0-5.0

---
## 2. Data Quality Assessment <a id='2-data-quality'></a>

In [ ]:
# Missing values analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing Count', ascending=False)

print("=" * 50)
print("MISSING VALUES ANALYSIS")
print("=" * 50)
missing_df[missing_df['Missing Count'] > 0]

In [ ]:
# Visualize missing values
fig, ax = plt.subplots(figsize=(10, 6))

missing_cols = missing_df[missing_df['Missing Count'] > 0]
if len(missing_cols) > 0:
    colors = ['#ff6b6b' if x > 0 else '#4ecdc4' for x in missing_cols['Missing Count']]
    ax.barh(missing_cols.index, missing_cols['Missing Count'], color=colors)
    ax.set_xlabel('Missing Count')
    ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold')
    
    for i, (idx, row) in enumerate(missing_cols.iterrows()):
        ax.text(row['Missing Count'] + 1, i, f"{row['Missing %']}%", va='center')
else:
    ax.text(0.5, 0.5, 'No Missing Values!', ha='center', va='center', fontsize=16)

plt.tight_layout()
plt.savefig('../reports/figures/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates} ({duplicates/len(df)*100:.2f}%)")

# Check unique values in categorical columns
print("\n" + "=" * 50)
print("CATEGORICAL FEATURES - UNIQUE VALUES")
print("=" * 50)

cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    if col != 'Order_ID':
        print(f"\n{col}: {df[col].nunique()} unique values")
        print(df[col].value_counts().head(5))

### Data Quality Findings:
- **Missing Values:** Only 0.1-0.2% in Agent_Rating and Weather (minimal)
- **Duplicates:** None found
- **Data Issues:** Some trailing spaces in categorical values (need cleaning)

---
## 3. Univariate Analysis <a id='3-univariate'></a>

In [ ]:
# Distribution of Delivery Time (Target Variable Basis)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['Delivery_Time'], bins=50, color='#FF9900', edgecolor='white', alpha=0.8)
axes[0].axvline(df['Delivery_Time'].mean(), color='red', linestyle='--', label=f"Mean: {df['Delivery_Time'].mean():.1f}")
axes[0].axvline(df['Delivery_Time'].median(), color='blue', linestyle='--', label=f"Median: {df['Delivery_Time'].median():.1f}")
axes[0].set_xlabel('Delivery Time (minutes)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Delivery Time', fontsize=12, fontweight='bold')
axes[0].legend()

# Box plot
axes[1].boxplot(df['Delivery_Time'], vert=True)
axes[1].set_ylabel('Delivery Time (minutes)')
axes[1].set_title('Delivery Time Box Plot', fontsize=12, fontweight='bold')

# Add percentile annotations
percentiles = [25, 50, 75, 90]
for p in percentiles:
    val = df['Delivery_Time'].quantile(p/100)
    axes[1].axhline(val, color='gray', linestyle=':', alpha=0.5)
    axes[1].text(1.1, val, f'P{p}: {val:.0f}', va='center')

plt.tight_layout()
plt.savefig('../reports/figures/delivery_time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nDelivery Time Statistics:")
print(f"  Mean: {df['Delivery_Time'].mean():.1f} minutes")
print(f"  Median: {df['Delivery_Time'].median():.1f} minutes")
print(f"  Std Dev: {df['Delivery_Time'].std():.1f} minutes")
print(f"  Min: {df['Delivery_Time'].min()} minutes")
print(f"  Max: {df['Delivery_Time'].max()} minutes")

In [ ]:
# Categorical Variables Distribution
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

cat_features = ['Weather', 'Traffic', 'Vehicle', 'Area', 'Category']
colors = ['#FF9900', '#232F3E', '#37475A', '#2ecc71', '#3498db']

for idx, (col, color) in enumerate(zip(cat_features, colors)):
    ax = axes[idx // 3, idx % 3]
    df[col].str.strip().value_counts().plot(kind='bar', ax=ax, color=color, edgecolor='white')
    ax.set_title(f'{col} Distribution', fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)

# Remove empty subplot
axes[1, 2].axis('off')

plt.tight_layout()
plt.savefig('../reports/figures/categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Agent Age and Rating Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Agent Age
axes[0].hist(df['Agent_Age'], bins=20, color='#232F3E', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Agent Age')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Agent Age', fontsize=12, fontweight='bold')

# Agent Rating
axes[1].hist(df['Agent_Rating'].dropna(), bins=20, color='#FF9900', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Agent Rating')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Agent Rating', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/agent_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Bivariate Analysis <a id='4-bivariate'></a>

In [ ]:
# Delivery Time by Weather Condition
fig, ax = plt.subplots(figsize=(12, 6))

df_clean = df.copy()
df_clean['Weather'] = df_clean['Weather'].str.strip()

weather_order = df_clean.groupby('Weather')['Delivery_Time'].mean().sort_values().index

sns.boxplot(data=df_clean, x='Weather', y='Delivery_Time', order=weather_order,
            palette='YlOrRd', ax=ax)
ax.set_title('Delivery Time by Weather Condition', fontsize=14, fontweight='bold')
ax.set_xlabel('Weather Condition')
ax.set_ylabel('Delivery Time (minutes)')

# Add mean annotations
means = df_clean.groupby('Weather')['Delivery_Time'].mean().reindex(weather_order)
for i, (weather, mean) in enumerate(means.items()):
    ax.text(i, mean + 5, f'{mean:.0f}', ha='center', fontweight='bold', color='darkred')

plt.tight_layout()
plt.savefig('../reports/figures/delivery_by_weather.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Delivery Time by Traffic Condition
fig, ax = plt.subplots(figsize=(10, 6))

df_clean['Traffic'] = df_clean['Traffic'].str.strip()
traffic_order = ['Low', 'Medium', 'High', 'Jam']

sns.violinplot(data=df_clean, x='Traffic', y='Delivery_Time', order=traffic_order,
               palette='RdYlGn_r', ax=ax)
ax.set_title('Delivery Time by Traffic Condition', fontsize=14, fontweight='bold')
ax.set_xlabel('Traffic Condition')
ax.set_ylabel('Delivery Time (minutes)')

plt.tight_layout()
plt.savefig('../reports/figures/delivery_by_traffic.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation Analysis
# Calculate distance first
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth's radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

df_clean['Distance_km'] = haversine_distance(
    df_clean['Store_Latitude'], df_clean['Store_Longitude'],
    df_clean['Drop_Latitude'], df_clean['Drop_Longitude']
)

# Correlation matrix
numeric_cols = ['Agent_Age', 'Agent_Rating', 'Distance_km', 'Delivery_Time']
corr_matrix = df_clean[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='RdYlGn', center=0,
            fmt='.3f', square=True, ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/correlation_matrix_eda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distance vs Delivery Time
fig, ax = plt.subplots(figsize=(10, 6))

# Sample for visualization
sample = df_clean.sample(min(5000, len(df_clean)), random_state=42)

scatter = ax.scatter(sample['Distance_km'], sample['Delivery_Time'],
                     c=sample['Delivery_Time'], cmap='RdYlGn_r',
                     alpha=0.5, s=20)

# Add trend line
z = np.polyfit(sample['Distance_km'], sample['Delivery_Time'], 1)
p = np.poly1d(z)
x_line = np.linspace(sample['Distance_km'].min(), sample['Distance_km'].max(), 100)
ax.plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Trend (slope={z[0]:.2f})')

ax.set_xlabel('Distance (km)')
ax.set_ylabel('Delivery Time (minutes)')
ax.set_title('Distance vs Delivery Time', fontsize=14, fontweight='bold')
ax.legend()

plt.colorbar(scatter, label='Delivery Time')
plt.tight_layout()
plt.savefig('../reports/figures/distance_vs_delivery.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Correlation between Distance and Delivery Time: {df_clean['Distance_km'].corr(df_clean['Delivery_Time']):.4f}")

---
## 5. Feature Engineering Preview <a id='5-feature-engineering'></a>

In [ ]:
# Extract temporal features
df_clean['Order_Date'] = pd.to_datetime(df_clean['Order_Date'])
df_clean['Order_Hour'] = pd.to_datetime(df_clean['Order_Time'], format='%H:%M:%S').dt.hour

df_clean['Day_of_Week'] = df_clean['Order_Date'].dt.dayofweek
df_clean['Is_Weekend'] = df_clean['Day_of_Week'].isin([5, 6]).astype(int)
df_clean['Is_Rush_Hour'] = df_clean['Order_Hour'].apply(
    lambda x: 1 if (8 <= x <= 10) or (17 <= x <= 19) else 0
)

# Time period
def get_time_period(hour):
    if 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df_clean['Time_Period'] = df_clean['Order_Hour'].apply(get_time_period)

In [ ]:
# Delivery Time by Hour of Day
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By hour
hourly = df_clean.groupby('Order_Hour')['Delivery_Time'].mean()
axes[0].plot(hourly.index, hourly.values, 'o-', color='#FF9900', linewidth=2, markersize=8)
axes[0].fill_between(hourly.index, hourly.values, alpha=0.3, color='#FF9900')
axes[0].axhline(df_clean['Delivery_Time'].mean(), color='red', linestyle='--', label='Overall Mean')

# Highlight rush hours
for hour in [8, 9, 10, 17, 18, 19]:
    if hour in hourly.index:
        axes[0].axvspan(hour-0.5, hour+0.5, alpha=0.2, color='red')

axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Average Delivery Time (minutes)')
axes[0].set_title('Delivery Time by Hour', fontsize=12, fontweight='bold')
axes[0].legend()

# By time period
time_order = ['Morning', 'Afternoon', 'Evening', 'Night']
period_mean = df_clean.groupby('Time_Period')['Delivery_Time'].mean().reindex(time_order)
colors = ['#ffd93d', '#ff9900', '#6c5ce7', '#2d3436']
axes[1].bar(period_mean.index, period_mean.values, color=colors, edgecolor='white')
axes[1].set_xlabel('Time Period')
axes[1].set_ylabel('Average Delivery Time (minutes)')
axes[1].set_title('Delivery Time by Time Period', fontsize=12, fontweight='bold')

for i, val in enumerate(period_mean.values):
    axes[1].text(i, val + 1, f'{val:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/delivery_by_time.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Target Variable Analysis <a id='6-target-analysis'></a>

In [ ]:
# Create target variable
def categorize_delivery(time):
    if time <= 90:
        return 'On-Time'
    elif time <= 160:
        return 'At-Risk'
    else:
        return 'Delayed'

df_clean['Delivery_Status'] = df_clean['Delivery_Time'].apply(categorize_delivery)

# Distribution
status_counts = df_clean['Delivery_Status'].value_counts()
status_pct = df_clean['Delivery_Status'].value_counts(normalize=True) * 100

print("=" * 50)
print("TARGET VARIABLE DISTRIBUTION")
print("=" * 50)
for status in ['On-Time', 'At-Risk', 'Delayed']:
    print(f"{status}: {status_counts[status]:,} ({status_pct[status]:.1f}%)")

In [ ]:
# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors = ['#28a745', '#ffc107', '#dc3545']
explode = [0.02, 0.02, 0.02]

axes[0].pie(status_counts.reindex(['On-Time', 'At-Risk', 'Delayed']),
            labels=['On-Time', 'At-Risk', 'Delayed'],
            colors=colors, explode=explode, autopct='%1.1f%%',
            shadow=True, startangle=90)
axes[0].set_title('Target Class Distribution', fontsize=14, fontweight='bold')

# Bar chart
status_order = ['On-Time', 'At-Risk', 'Delayed']
bars = axes[1].bar(status_order, status_counts.reindex(status_order), color=colors, edgecolor='white')
axes[1].set_ylabel('Count')
axes[1].set_title('Target Class Counts', fontsize=14, fontweight='bold')

for bar, count in zip(bars, status_counts.reindex(status_order)):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{count:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/target_distribution_eda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Class imbalance note
print("\n⚠️ CLASS IMBALANCE DETECTED")
print("="*50)
print("The 'At-Risk' class dominates (~49%), while 'On-Time' (~28%)")
print("and 'Delayed' (~23%) are underrepresented.")
print("\n✅ SOLUTION: We will use SMOTE (Synthetic Minority")
print("Over-sampling Technique) to balance the classes.")

---
## 7. Key Insights & Findings <a id='7-insights'></a>

### Summary of Key Findings

#### 1. Data Quality
- Dataset is relatively clean with minimal missing values (<0.3%)
- No duplicate records found
- Some categorical values need string cleaning (trailing spaces)

#### 2. Target Variable Insights
- **28.1%** deliveries are On-Time (≤90 min)
- **48.8%** deliveries are At-Risk (91-160 min)
- **23.1%** deliveries are Delayed (>160 min)
- Class imbalance exists → SMOTE required

#### 3. Key Predictive Factors
| Factor | Impact on Delay |
|--------|----------------|
| **Distance** | Strong positive correlation |
| **Weather** | Stormy/Sandstorms increase delays |
| **Traffic** | Jam conditions significantly impact |
| **Rush Hour** | Peak hours show higher delay rates |
| **Agent Rating** | Higher ratings correlate with faster delivery |

#### 4. Business Recommendations
1. **Proactive Alerts**: Flag high-risk deliveries at order time
2. **Weather Integration**: Adjust ETAs based on weather forecasts
3. **Route Optimization**: Consider traffic patterns in scheduling
4. **Agent Allocation**: Assign experienced agents to high-risk orders

---

### Next Steps
1. Data Preprocessing Pipeline
2. Feature Engineering (19 features)
3. Model Training & Evaluation
4. Dashboard Development

In [ ]:
print("\n" + "="*60)
print("EDA COMPLETE!")
print("="*60)
print(f"\nDataset: 43,739 Amazon delivery records")
print(f"Features analyzed: 16 original + engineered")
print(f"Target: 3-class classification (On-Time, At-Risk, Delayed)")
print(f"\nAll visualizations saved to: ../reports/figures/")